# Track A — Bước A6 (Ưu tiên #1): Cộng hưởng `DINOv3` + `Imbalanced-DYDM` + `NIW Memory` trên `GC10-DET`

- Nhánh: `duytung/10-gc10-dinov3-imbal-dydm`
- Tái sử dụng trực tiếp checkpoint `DINOv3` (`ft_ckpt.pt`, `FT_EPOCHS=20`) và 5 snapshot `s0..s4` của Bước A2b trên Drive (không cần chạy lại Phase 1!).
- Đánh giá đủ `N`, `NL`, `FTR1`, `D3_Imb_head`, `D3_Imb_km`, `D3_Imb_blend`, `D3_Imb_blend_N` và xuất `gc10_a6_dinov3_dydm_summary.txt` / `.json`.

In [ ]:
# 1. Kết nối Google Drive (bỏ qua nếu runtime hiện tại đã mount sẵn)
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Drive đã được mount sẵn!')

In [ ]:
# 2. Cập nhật nhánh duytung/10-gc10-dinov3-imbal-dydm
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
BRANCH = 'duytung/10-gc10-dinov3-imbal-dydm'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git status -sb

In [ ]:
# 3. Cài gói thiếu (bỏ qua nếu runtime hiện tại đã cài)
!pip uninstall -y -q torchao
!pip install -q numpyro continuum peft datasets transformers scipy scikit-learn psutil
!nvidia-smi

In [ ]:
# 4. Nạp HF_TOKEN và chạy nền Bước A6 (DINOv3 + Imbalanced-DYDM, 5 seeds s0..s4)
import os, subprocess, time
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a6_dinov3_dydm.log'
PID_FILE = f'{WORK_DIR}/gc10_a6_dinov3_dydm.pid'

hf_token = os.environ.get('HF_TOKEN', '')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or ''
    except Exception:
        hf_token = ''
# Nếu chưa lưu Colab Secrets, bỏ dấu # dòng dưới và dán token tạm:
# hf_token = 'hf_...'

env = os.environ.copy()
if hf_token:
    env['HF_TOKEN'] = hf_token
with open(LOG_FILE, 'w') as f:
    p = subprocess.Popen(['bash', 'colab/run_gc10_a6_bg.sh', 'auto'], stdout=f, stderr=subprocess.STDOUT, env=env)
with open(PID_FILE, 'w') as f:
    f.write(str(p.pid))
print(f'Đã khởi chạy nền Bước A6 (DINOv3 + Imbalanced-DYDM) với PID={p.pid}')
print(f'Log: {LOG_FILE}')
time.sleep(3)
!tail -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ & xem kết quả + thông số phần cứng peak
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a6_dinov3_dydm.log'
PID_FILE = f'{WORK_DIR}/gc10_a6_dinov3_dydm.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a6_dinov3_dydm_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}: {"ĐANG CHẠY" if alive else "ĐÃ DỪNG / XONG"}')
print('\n--- 40 dòng log gần nhất ---')
if os.path.exists(LOG_FILE):
    lines = open(LOG_FILE).read().splitlines()
    print('\n'.join(lines[-40:]))
if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG PEAK BƯỚC A6 (GC10 DINOv3 + Imbalanced-DYDM) ---')
    print(open(SUMMARY_OUT).read())